<a href="https://colab.research.google.com/github/Keerthan-AC-583/BIS-Lab/blob/main/1BM24CS133_GA_5G_network_allocation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# ============================================================
# 5G NETWORK RESOURCE ALLOCATION USING GENETIC ALGORITHM
# ============================================================

# -----------------------------
# 1. SYSTEM PARAMETERS
# -----------------------------

NUM_USERS = 5

# Total available resources
TOTAL_POWER = 10.0          # Watts
TOTAL_BANDWIDTH = 20e6      # 20 MHz

# Noise power
NOISE_POWER = 1e-9

# Channel gains for each user
CHANNEL_GAIN = np.array([
    0.80,
    0.60,
    0.40,
    0.70,
    0.50
])

# Minimum required data rate for each user
MIN_RATE = np.array([
    1e6,
    1e6,
    1e6,
    1e6,
    1e6
])


# ============================================================
# 2. GENETIC ALGORITHM PARAMETERS
# ============================================================

POPULATION_SIZE = 50
MAX_GENERATIONS = 100

CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.1

ELITE_SIZE = 2

# Penalty coefficients
POWER_PENALTY = 10
BANDWIDTH_PENALTY = 10
QOS_PENALTY = 5

CHROMOSOME_LENGTH = NUM_USERS * 2


# ============================================================
# 3. INITIALIZE POPULATION
# ============================================================

def initialize_population():

    population = []

    for _ in range(POPULATION_SIZE):

        # Random power allocation
        power = np.random.uniform(
            0.1,
            TOTAL_POWER / NUM_USERS,
            NUM_USERS
        )

        # Random bandwidth allocation
        bandwidth = np.random.uniform(
            1e6,
            TOTAL_BANDWIDTH / NUM_USERS,
            NUM_USERS
        )

        # Create chromosome
        chromosome = np.concatenate(
            (power, bandwidth)
        )

        population.append(chromosome)

    return np.array(population)


# ============================================================
# 4. NORMALIZE RESOURCES
# ============================================================

def normalize_resources(chromosome):

    power = chromosome[:NUM_USERS]
    bandwidth = chromosome[NUM_USERS:]

    # Make values positive
    power = np.maximum(power, 0.01)
    bandwidth = np.maximum(bandwidth, 1e5)

    # Normalize power
    if np.sum(power) > TOTAL_POWER:

        power = (
            power / np.sum(power)
        ) * TOTAL_POWER

    # Normalize bandwidth
    if np.sum(bandwidth) > TOTAL_BANDWIDTH:

        bandwidth = (
            bandwidth / np.sum(bandwidth)
        ) * TOTAL_BANDWIDTH

    return np.concatenate(
        (power, bandwidth)
    )


# ============================================================
# 5. CALCULATE DATA RATE
# ============================================================

def calculate_data_rate(power, bandwidth):

    # Shannon capacity:
    #
    # R = B * log2(1 + P*h/N)

    snr = (
        power * CHANNEL_GAIN
    ) / NOISE_POWER

    rate = (
        bandwidth *
        np.log2(1 + snr)
    )

    return rate


# ============================================================
# 6. CALCULATE FAIRNESS
# ============================================================

def calculate_fairness(rates):

    numerator = np.sum(rates) ** 2

    denominator = (
        NUM_USERS *
        np.sum(rates ** 2)
    )

    if denominator == 0:
        return 0

    fairness = (
        numerator / denominator
    )

    return fairness


# ============================================================
# 7. FITNESS FUNCTION
# ============================================================

def calculate_fitness(chromosome):

    chromosome = normalize_resources(
        chromosome
    )

    power = chromosome[:NUM_USERS]
    bandwidth = chromosome[NUM_USERS:]

    # Calculate data rates
    rates = calculate_data_rate(
        power,
        bandwidth
    )

    # Total throughput
    total_throughput = np.sum(rates)

    # Energy efficiency
    total_power = np.sum(power)

    energy_efficiency = (
        total_throughput /
        total_power
    )

    # Fairness
    fairness = calculate_fairness(
        rates
    )

    # ------------------------------------
    # QoS penalty
    # ------------------------------------

    qos_penalty = 0

    for i in range(NUM_USERS):

        if rates[i] < MIN_RATE[i]:

            qos_penalty += (
                (MIN_RATE[i] - rates[i])
                / MIN_RATE[i]
            )

    # ------------------------------------
    # Resource penalties
    # ------------------------------------

    power_penalty = max(
        0,
        np.sum(power) - TOTAL_POWER
    )

    bandwidth_penalty = max(
        0,
        np.sum(bandwidth) -
        TOTAL_BANDWIDTH
    )

    # ------------------------------------
    # Fitness
    # ------------------------------------

    fitness = (

        total_throughput / 1e6

        + fairness * 10

        + energy_efficiency / 1e6

        - QOS_PENALTY * qos_penalty

        - POWER_PENALTY * power_penalty

        - BANDWIDTH_PENALTY *
        (bandwidth_penalty / 1e6)
    )

    return fitness


# ============================================================
# 8. SELECTION
# ============================================================

def selection(population, fitness_values):

    selected = []

    for _ in range(POPULATION_SIZE):

        # Select 3 random chromosomes
        candidates = np.random.choice(
            POPULATION_SIZE,
            3,
            replace=False
        )

        # Select the best one
        best_index = candidates[
            np.argmax(
                fitness_values[candidates]
            )
        ]

        selected.append(
            population[best_index]
        )

    return np.array(selected)


# ============================================================
# 9. CROSSOVER
# ============================================================

def crossover(parent1, parent2):

    child1 = parent1.copy()
    child2 = parent2.copy()

    if np.random.rand() < CROSSOVER_RATE:

        # Select crossover point
        point = np.random.randint(
            1,
            CHROMOSOME_LENGTH
        )

        # Create children
        child1[:point] = parent1[:point]
        child1[point:] = parent2[point:]

        child2[:point] = parent2[:point]
        child2[point:] = parent1[point:]

    return child1, child2


# ============================================================
# 10. MUTATION
# ============================================================

def mutation(chromosome):

    chromosome = chromosome.copy()

    for i in range(
        CHROMOSOME_LENGTH
    ):

        if np.random.rand() < MUTATION_RATE:

            # Power mutation
            if i < NUM_USERS:

                chromosome[i] += (
                    np.random.normal(0, 0.5)
                )

                chromosome[i] = max(
                    chromosome[i],
                    0.01
                )

            # Bandwidth mutation
            else:

                chromosome[i] += (
                    np.random.normal(0, 1e6)
                )

                chromosome[i] = max(
                    chromosome[i],
                    1e5
                )

    return chromosome


# ============================================================
# 11. GENETIC ALGORITHM
# ============================================================

def genetic_algorithm():

    # Initialize population
    population = initialize_population()

    best_solution = None
    best_fitness = -np.inf

    # ------------------------------------
    # Main GA loop
    # ------------------------------------

    for generation in range(
        MAX_GENERATIONS
    ):

        # Calculate fitness
        fitness_values = np.array([
            calculate_fitness(chromosome)
            for chromosome in population
        ])

        # Find best solution
        current_best_index = np.argmax(
            fitness_values
        )

        current_best_fitness = (
            fitness_values[
                current_best_index
            ]
        )

        current_best_solution = (
            population[
                current_best_index
            ].copy()
        )

        # Update global best
        if (
            current_best_fitness >
            best_fitness
        ):

            best_fitness = (
                current_best_fitness
            )

            best_solution = (
                current_best_solution.copy()
            )

        # --------------------------------
        # Selection
        # --------------------------------

        selected_population = selection(
            population,
            fitness_values
        )

        # --------------------------------
        # Elitism
        # --------------------------------

        elite_indices = np.argsort(
            fitness_values
        )[-ELITE_SIZE:]

        new_population = [
            population[i].copy()
            for i in elite_indices
        ]

        # --------------------------------
        # Crossover + Mutation
        # --------------------------------

        while len(new_population) < POPULATION_SIZE:

            parent_indices = np.random.choice(
                POPULATION_SIZE,
                2,
                replace=False
            )

            parent1 = selected_population[
                parent_indices[0]
            ]

            parent2 = selected_population[
                parent_indices[1]
            ]

            # Crossover
            child1, child2 = crossover(
                parent1,
                parent2
            )

            # Mutation
            child1 = mutation(child1)
            child2 = mutation(child2)

            # Normalize
            child1 = normalize_resources(
                child1
            )

            child2 = normalize_resources(
                child2
            )

            new_population.append(
                child1
            )

            if (
                len(new_population) <
                POPULATION_SIZE
            ):

                new_population.append(
                    child2
                )

        population = np.array(
            new_population
        )

    return (
        best_solution,
        best_fitness
    )


# ============================================================
# 12. RUN GENETIC ALGORITHM
# ============================================================

best_solution, best_fitness = (
    genetic_algorithm()
)


# ============================================================
# 13. GET BEST RESOURCE ALLOCATION
# ============================================================

best_solution = normalize_resources(
    best_solution
)

best_power = (
    best_solution[:NUM_USERS]
)

best_bandwidth = (
    best_solution[NUM_USERS:]
)


# ============================================================
# 14. CALCULATE FINAL PERFORMANCE
# ============================================================

best_rates = calculate_data_rate(
    best_power,
    best_bandwidth
)

total_throughput = np.sum(
    best_rates
)

fairness = calculate_fairness(
    best_rates
)

energy_efficiency = (
    total_throughput /
    np.sum(best_power)
)


# ============================================================
# 15. DISPLAY FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("       5G NETWORK RESOURCE ALLOCATION")
print("       USING GENETIC ALGORITHM")
print("=" * 70)

print(
    "\nBest Fitness :",
    round(best_fitness, 4)
)

print("\nResource Allocation:")
print("-" * 70)

print(
    f"{'User':<10}"
    f"{'Power (W)':<15}"
    f"{'Bandwidth (MHz)':<20}"
    f"{'Data Rate (Mbps)':<20}"
)

print("-" * 70)

for i in range(NUM_USERS):

    print(
        f"User {i + 1:<5}"
        f"{best_power[i]:<15.4f}"
        f"{best_bandwidth[i] / 1e6:<20.4f}"
        f"{best_rates[i] / 1e6:<20.4f}"
    )

print("-" * 70)

print(
    f"\nTotal Power       : "
    f"{np.sum(best_power):.4f} W"
)

print(
    f"Total Bandwidth   : "
    f"{np.sum(best_bandwidth) / 1e6:.4f} MHz"
)

print(
    f"Total Throughput  : "
    f"{total_throughput / 1e6:.4f} Mbps"
)

print(
    f"Fairness Index    : "
    f"{fairness:.4f}"
)

print(
    f"Energy Efficiency : "
    f"{energy_efficiency / 1e6:.4f} Mbps/W"
)

print("=" * 70)



       5G NETWORK RESOURCE ALLOCATION
       USING GENETIC ALGORITHM

Best Fitness : 9603.718

Resource Allocation:
----------------------------------------------------------------------
User      Power (W)      Bandwidth (MHz)     Data Rate (Mbps)    
----------------------------------------------------------------------
User 1    0.0100         13.0597             299.4790            
User 2    0.0100         0.0999              2.2493              
User 3    0.0100         0.0999              2.1909              
User 4    0.0100         6.6406              151.0007            
User 5    0.0100         0.0999              2.2230              
----------------------------------------------------------------------

Total Power       : 0.0500 W
Total Bandwidth   : 20.0000 MHz
Total Throughput  : 457.1430 Mbps
Fairness Index    : 0.3715
Energy Efficiency : 9142.8600 Mbps/W
